In [5]:
import gradio as gr
import requests
import io
from PIL import Image
import os
from datetime import datetime

# =========================
# API CONFIG
# =========================

API_URL = "https://router.huggingface.co/hf-inference/models/black-forest-labs/FLUX.1-schnell"




HF_TOKEN = os.getenv("HF_TOKEN")

headers = {
    "Authorization": f"Bearer {HF_TOKEN}"
}

# =========================
# STYLE PRESETS
# =========================

STYLE_MAP = {
    "None": "",
    "Cinematic": "cinematic lighting, ultra realistic, dramatic shadows, movie still, masterpiece",
    "Cyberpunk": "cyberpunk, neon lights, futuristic city, synthwave aesthetic",
    "Fantasy": "fantasy art, magical atmosphere, detailed digital painting",
    "Anime": "anime style, vibrant colors, studio ghibli quality",
    "Studio Photo": "professional photography, sharp focus, studio lighting",
    "3D Render": "3d render, octane render, unreal engine, hyper detailed"
}

NEGATIVE_DEFAULT = (
    "blurry, low quality, distorted face, bad anatomy, "
    "extra fingers, cropped, watermark, text"
)

# =========================
# IMAGE GENERATION
# =========================

def generate_image(
    prompt,
    negative_prompt,
    style,
    steps
):

    full_prompt = f"""
    {prompt},
    {STYLE_MAP[style]}
    Negative prompt: {negative_prompt}
    """

    payload = {
        "inputs": full_prompt,
        "parameters": {
            "num_inference_steps": int(steps)
        }
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload,
        timeout=120
    )

    if response.status_code != 200:
        raise gr.Error(
            f"API Error {response.status_code}: {response.text[:200]}"
        )

    image = Image.open(io.BytesIO(response.content))

    return image


# =========================
# CUSTOM CSS
# =========================

custom_css = """

body {
    background: #020617;
}

.gradio-container {
    max-width: 1400px !important;
}

#main-wrapper {
    border-radius: 20px;
    padding: 25px;
    background: linear-gradient(
        145deg,
        #0f172a,
        #111827
    );
    border: 1px solid #1e293b;
}

.main-title {
    text-align: center;
    font-size: 42px;
    font-weight: bold;
    margin-bottom: 10px;
    background: linear-gradient(to right, #8b5cf6, #3b82f6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.subtitle {
    text-align: center;
    color: #94a3b8;
    margin-bottom: 30px;
}

.generate-btn {
    height: 55px;
    font-size: 18px !important;
    border-radius: 14px !important;
    background: linear-gradient(
        90deg,
        #7c3aed,
        #2563eb
    ) !important;
    border: none !important;
}

footer {
    display: none !important;
}

"""

# =========================
# UI
# =========================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        neutral_hue="slate"
    ),
    css=custom_css
) as demo:

    with gr.Column(elem_id="main-wrapper"):

        gr.HTML("""
        <div class="main-title">
            🌌 Visionary AI Studio
        </div>

        <div class="subtitle">
            Generate professional AI images with FLUX Schnell
        </div>
        """)

        with gr.Row():

            # =====================
            # LEFT PANEL
            # =====================

            with gr.Column(scale=1):

                prompt_input = gr.Textbox(
                    label="Prompt",
                    placeholder="Describe your image...",
                    lines=5
                )

                negative_prompt = gr.Textbox(
                    label="Negative Prompt",
                    value=NEGATIVE_DEFAULT,
                    lines=3
                )

                style_input = gr.Dropdown(
                    choices=list(STYLE_MAP.keys()),
                    value="Cinematic",
                    label="Art Style"
                )

                with gr.Accordion("Advanced Settings", open=False):

                    steps_slider = gr.Slider(
                        label="Inference Steps",
                        minimum=1,
                        maximum=4,
                        value=4,
                        step=1
                    )

                generate_btn = gr.Button(
                    "✨ Generate Image",
                    elem_classes="generate-btn",
                    variant="primary"
                )

                gr.Examples(
                    examples=[
                        [
                            "A futuristic cyberpunk city at night with flying cars"
                        ],
                        [
                            "A lion king sitting on a golden throne"
                        ],
                        [
                            "A cinematic portrait of a warrior woman"
                        ],
                        [
                            "A luxury modern bedroom interior design"
                        ]
                    ],
                    inputs=prompt_input
                )

            # =====================
            # RIGHT PANEL
            # =====================

            with gr.Column(scale=1):

                image_output = gr.Image(
                    label="Generated Image",
                    type="pil",
                    height=650
                )

                download_btn = gr.DownloadButton(
                    label="⬇ Download Image",
                    visible=False
                )

    # =========================
    # GENERATE EVENT
    # =========================

    def generate_and_download(
        prompt,
        negative_prompt,
        style,
        steps
    ):

        image = generate_image(
            prompt,
            negative_prompt,
            style,
            steps
        )

        filename = f"visionary_{datetime.now().strftime('%H%M%S')}.png"

        image.save(filename)

        return image, filename, gr.update(visible=True)

    generate_btn.click(
        fn=generate_and_download,
        inputs=[
            prompt_input,
            negative_prompt,
            style_input,
            steps_slider
        ],
        outputs=[
            image_output,
            download_btn,
            download_btn
        ]
    )

# =========================
# LAUNCH
# =========================

demo.queue()
demo.launch(debug=True)

/tmp/ipykernel_1903/1029621891.py:146: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_1903/1029621891.py:146: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3b4037c1e15a21446a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3b4037c1e15a21446a.gradio.live
